In [1]:
import argparse
import os
import pathlib
import sys

import numpy as np
import pandas as pd
from image_analysis_3D.featurization_utils.feature_writing_utils import (
    format_morphology_feature_name,
)
from image_analysis_3D.featurization_utils.neighbors_utils import (
    classify_cells_into_shells,
    euclidean_distance_from_centroid,
    mahalanobis_distance_from_centroid,
)
from image_analysis_3D.file_utils.arg_parsing_utils import parse_args
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)
from tqdm import tqdm

root_dir, in_notebook = init_notebook()

profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot")).resolve(), root_dir
)

In [2]:
if not in_notebook:
    args = parse_args()
    well_fov = args["well_fov"]
    patient = args["patient"]
    image_based_profiles_subparent_name = args["image_based_profiles_subparent_name"]

else:
    patient = "NF0014_T1"
    well_fov = "C4-2"
    image_based_profiles_subparent_name = "image_based_profiles"

In [3]:
def centroid_within_bbox_detection(
    centroid: tuple,
    bbox: tuple,
) -> bool:
    """
    Check if the centroid is within the bbox

    Parameters
    ----------
    centroid : tuple
        Centroid of the object in the order of (z, y, x)
        Order of the centroid is important
    bbox : tuple
        Where the bbox is in the order of (z_min, y_min, x_min, z_max, y_max, x_max)
        Order of the bbox is important

    Returns
    -------
    bool
        True if the centroid is within the bbox, False otherwise
    """
    z_min, y_min, x_min, z_max, y_max, x_max = bbox
    z, y, x = centroid
    # check if the centroid is within the bbox
    if (
        z >= z_min
        and z <= z_max
        and y >= y_min
        and y <= y_max
        and x >= x_min
        and x <= x_max
    ):
        return True
    else:
        return False

### Pathing

In [4]:
# input paths
sc_profile_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/sc_profiles_{well_fov}.parquet"
).resolve(strict=True)
organoid_profile_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/organoid_profiles_{well_fov}.parquet"
).resolve(strict=True)
nucleocentric_profile_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/nucleocentric_profiles_{well_fov}.parquet"
).resolve(strict=True)
# output paths
sc_profile_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/1.related_profiles/{well_fov}/sc_profiles_{well_fov}_related.parquet"
).resolve()
organoid_profile_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/1.related_profiles/{well_fov}/organoid_profiles_{well_fov}_related.parquet"
).resolve()
nucleocentric_profile_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/1.related_profiles/{well_fov}/nucleocentric_profiles_{well_fov}_related.parquet"
).resolve()
sc_profile_output_path.parent.mkdir(parents=True, exist_ok=True)

In [5]:
sc_profile_df = pd.read_parquet(sc_profile_path)
nucleocentric_df = pd.read_parquet(nucleocentric_profile_path)
organoid_profile_df = pd.read_parquet(organoid_profile_path)
print(f"Single-cell profile shape: {sc_profile_df.shape}")
print(f"Nucleocentric profile shape: {nucleocentric_df.shape}")
print(f"Organoid profile shape: {organoid_profile_df.shape}")

Single-cell profile shape: (41, 10011)
Nucleocentric profile shape: (41, 3074)
Organoid profile shape: (1, 3337)


In [6]:
# initialize the parent organoid column
sc_profile_df.insert(2, "ParentOrganoid", -1)

In [7]:
x_y_z_sc_colnames = [
    x
    for x in sc_profile_df.columns
    if "area" in x.lower() and "center" in x.lower() and "nuclei" in x.lower()
]
x_y_z_sc_colnames

['Nuclei_NoChannel_AreaSizeShape_CenterX',
 'Nuclei_NoChannel_AreaSizeShape_CenterY',
 'Nuclei_NoChannel_AreaSizeShape_CenterZ']

In [8]:
organoid_bbox_colnames = [
    x
    for x in organoid_profile_df.columns
    if "area" in x.lower() and ("min" in x.lower() or "max" in x.lower())
]
organoid_bbox_colnames = sorted(organoid_bbox_colnames)

In [9]:
sc_centroids = sc_profile_df[
    x_y_z_sc_colnames
].values  # alphabetically sorted to be in the order of x,y,z

In [10]:
# Initialize parent_organoid to -1
sc_profile_df["ParentOrganoid"] = -1

# Extract single-cell centroids as numpy array for faster access
sc_centroids = sc_profile_df[x_y_z_sc_colnames].values  # (N_cells, 3) array
# reshape the centroids to be in z,y,x order for easier comparison with bbox
sc_centroids = sc_centroids[:, [2, 1, 0]]  # reorder to z,y,x

# Loop through organoids with progress bar
for organoid_index, organoid_row in tqdm(
    organoid_profile_df.iterrows(),
    total=len(organoid_profile_df),
    desc="Assigning cells to organoids",
):
    # Get organoid bbox
    organoid_bbox = (
        organoid_row[organoid_bbox_colnames[5]],  # z_min
        organoid_row[organoid_bbox_colnames[4]],  # y_min
        organoid_row[organoid_bbox_colnames[3]],  # x_min
        organoid_row[organoid_bbox_colnames[2]],  # z_max
        organoid_row[organoid_bbox_colnames[1]],  # y_max
        organoid_row[organoid_bbox_colnames[0]],  # x_max
    )

    z_min, y_min, x_min, z_max, y_max, x_max = organoid_bbox

    # Vectorized bbox check - much faster!
    mask = (
        (sc_centroids[:, 0] >= z_min)  # z
        & (sc_centroids[:, 0] <= z_max)  # z
        & (sc_centroids[:, 1] >= y_min)
        & (sc_centroids[:, 1] <= y_max)
        & (sc_centroids[:, 2] >= x_min)
        & (sc_centroids[:, 2] <= x_max)
    )

    # Only assign if cell doesn't already have a parent
    unassigned_mask = sc_profile_df["ParentOrganoid"] == -1
    final_mask = mask & unassigned_mask

    # Assign parent organoid to matching cells
    sc_profile_df.loc[final_mask, "ParentOrganoid"] = organoid_row["object_id"]

print(f"Assigned {(sc_profile_df['ParentOrganoid'] != -1).sum()} cells to organoids")
print(f"Unassigned cells: {(sc_profile_df['ParentOrganoid'] == -1).sum()}")

Assigning cells to organoids: 100%|██████████| 1/1 [00:00<00:00, 499.56it/s]

Assigned 41 cells to organoids
Unassigned cells: 0


### Add single-cell counts for each organoid

In [13]:
organoid_sc_counts = (
    sc_profile_df["ParentOrganoid"]
    .value_counts()
    .to_frame(name="OrganoidSingleCellCount")
    .reset_index()
)
# merge the organoid profile with the single-cell counts
organoid_profile_df = pd.merge(
    organoid_profile_df,
    organoid_sc_counts,
    left_on="object_id",
    right_on="ParentOrganoid",
    how="left",
).drop(columns=["ParentOrganoid"])
sc_count = organoid_profile_df.pop("OrganoidSingleCellCount")
organoid_profile_df.insert(2, "OrganoidSingleCellCount", sc_count)

Even if the file is empty we still want to add it to the final dataframe dictionary so that we can merge on the same columns later.
This will help with file-based checking and merging.


In [14]:
# replace NaN with 0 for organoids that have no assigned cells
organoid_profile_df["OrganoidSingleCellCount"] = (
    organoid_profile_df["OrganoidSingleCellCount"].fillna(0).astype(int)
)
organoid_profile_df.head()

,object_id,image_set,OrganoidSingleCellCount,SingleCellCount,Organoid_NoChannel_AreaSizeShape_Volume,Organoid_NoChannel_AreaSizeShape_CenterX,Organoid_NoChannel_AreaSizeShape_CenterY,Organoid_NoChannel_AreaSizeShape_CenterZ,Organoid_NoChannel_AreaSizeShape_BboxVolume,Organoid_NoChannel_AreaSizeShape_MinX,...,Organoid_Mito_Texture_DifferenceEntropy-256-3,Organoid_Mito_Texture_DifferenceVariance-256-3,Organoid_Mito_Texture_Entropy-256-3,Organoid_Mito_Texture_InformationMeasureOfCorrelation1-256-3,Organoid_Mito_Texture_InformationMeasureOfCorrelation2-256-3,Organoid_Mito_Texture_InverseDifferenceMoment-256-3,Organoid_Mito_Texture_SumAverage-256-3,Organoid_Mito_Texture_SumEntropy-256-3,Organoid_Mito_Texture_SumVariance-256-3,Organoid_Mito_Texture_Variance-256-3
0,1,C4-2,41,41,18671184.0,671.803338,563.720671,14.722934,30119463.0,229,...,1.307112,0.002466,2.531704,-0.486536,0.89619,0.839941,7.675781,1.927611,193.311708,49.2295


In [15]:
if organoid_profile_df.empty:
    # add a row with 0 values
    organoid_profile_df.loc[len(organoid_profile_df)] = [0] * len(
        organoid_profile_df.columns
    )
    organoid_profile_df["image_set"] = well_fov

In [16]:
print(f"Single-cell profile shape: {sc_profile_df.shape}")

Single-cell profile shape: (41, 10012)


In [17]:
if sc_profile_df.empty:
    # add a row with Na values
    sc_profile_df.loc[len(sc_profile_df)] = [None] * len(sc_profile_df.columns)
    sc_profile_df["image_set"] = well_fov

In [18]:
# add the parent organoid to nucleocentric features
nucleocentric_df = pd.merge(
    nucleocentric_df,
    sc_profile_df[["object_id", "image_set", "ParentOrganoid"]],
    on=["object_id", "image_set"],
    how="left",
)

## Get single cell and organoid relationships and spatial distributions

In [19]:
x_y_z_organoid_centroid_colnames = [
    x
    for x in organoid_profile_df.columns
    if "area" in x.lower() and "center" in x.lower()
]
x_y_z_organoid_bbox_colnames = [
    x
    for x in organoid_profile_df.columns
    if "area" in x.lower() and ("min" in x.lower() or "max" in x.lower())
]

In [20]:
results = []

# get the organoid id and the single-cells for each

organoid_ids = organoid_profile_df["object_id"]

# organoid_id = organoid_ids[0]

for organoid_id in organoid_ids:
    organoid_centroid = (
        organoid_profile_df.loc[
            organoid_profile_df["object_id"] == organoid_id,
            x_y_z_organoid_centroid_colnames,
        ]
        .apply(pd.to_numeric, errors="coerce")
        .iloc[0]
        .to_numpy(dtype=float)
    )
    organoid_bbox = organoid_profile_df.loc[
        organoid_profile_df["object_id"] == organoid_id, x_y_z_organoid_bbox_colnames
    ].values[0]
    single_cells_in_organoid = sc_profile_df[
        sc_profile_df["ParentOrganoid"] == organoid_id
    ]
    if single_cells_in_organoid.empty:
        print(f"No single cells assigned to organoid {organoid_id}")
        continue

    single_cells_centroids = (
        single_cells_in_organoid[x_y_z_sc_colnames]
        .apply(pd.to_numeric, errors="coerce")
        .to_numpy(dtype=float)
    )

    valid_rows = ~pd.isna(single_cells_centroids).any(axis=1)
    single_cells_centroids = single_cells_centroids[valid_rows]
    single_cells_in_organoid = single_cells_in_organoid.loc[valid_rows]

    if single_cells_centroids.shape[0] == 0:
        continue

    # convert to a dict with the key being the object_id
    # rename the centroids to z,y.x
    single_cells_centroids_dict = {
        "object_id": single_cells_in_organoid["object_id"].to_numpy(),
        "z": single_cells_in_organoid[x_y_z_sc_colnames[2]].to_numpy(dtype=float),
        "y": single_cells_in_organoid[x_y_z_sc_colnames[1]].to_numpy(dtype=float),
        "x": single_cells_in_organoid[x_y_z_sc_colnames[0]].to_numpy(dtype=float),
    }

    euclidean_distance = euclidean_distance_from_centroid(
        single_cells_centroids, organoid_centroid
    )
    mahalanobis_distance = mahalanobis_distance_from_centroid(
        single_cells_centroids, organoid_centroid
    )
    shell_classification, centroid = classify_cells_into_shells(
        coords=single_cells_centroids_dict,
        n_shells=4,
        method="mahalanobis",
        min_cells_per_shell=3,
        centroid=organoid_centroid,
    )

    shell_classification_df = pd.DataFrame(shell_classification)
    shell_classification_df["ParentOrganoid"] = organoid_id
    results.append(shell_classification_df)

In [21]:
if results:
    df = pd.concat([pd.DataFrame(r) for r in results], ignore_index=True)

else:
    df = pd.DataFrame(columns=["object_id", "ParentOrganoid"])

# rename the columns

df.rename(
    columns={
        col: format_morphology_feature_name(
            compartment="Nuclei",
            feature_type="Neighbors",
            channel="NoChannel",
            measurement=col,
        )
        for col in df.columns
        if col not in ["object_id", "ParentOrganoid"]
    },
    inplace=True,
)

In [22]:
# concat the shell classification with the single cell profile df to get the full single cell profile with the shell classification and the parent organoid id
sc_profile_with_shells_df = pd.merge(
    sc_profile_df,
    df,
    left_on=["object_id", "ParentOrganoid"],
    right_on=["object_id", "ParentOrganoid"],
    how="left",
)

### Save the profiles

In [23]:
organoid_profile_df.to_parquet(organoid_profile_output_path, index=False)
organoid_profile_df.head()

,object_id,image_set,OrganoidSingleCellCount,SingleCellCount,Organoid_NoChannel_AreaSizeShape_Volume,Organoid_NoChannel_AreaSizeShape_CenterX,Organoid_NoChannel_AreaSizeShape_CenterY,Organoid_NoChannel_AreaSizeShape_CenterZ,Organoid_NoChannel_AreaSizeShape_BboxVolume,Organoid_NoChannel_AreaSizeShape_MinX,...,Organoid_Mito_Texture_DifferenceEntropy-256-3,Organoid_Mito_Texture_DifferenceVariance-256-3,Organoid_Mito_Texture_Entropy-256-3,Organoid_Mito_Texture_InformationMeasureOfCorrelation1-256-3,Organoid_Mito_Texture_InformationMeasureOfCorrelation2-256-3,Organoid_Mito_Texture_InverseDifferenceMoment-256-3,Organoid_Mito_Texture_SumAverage-256-3,Organoid_Mito_Texture_SumEntropy-256-3,Organoid_Mito_Texture_SumVariance-256-3,Organoid_Mito_Texture_Variance-256-3
0,1,C4-2,41,41,18671184.0,671.803338,563.720671,14.722934,30119463.0,229,...,1.307112,0.002466,2.531704,-0.486536,0.89619,0.839941,7.675781,1.927611,193.311708,49.2295


In [24]:
sc_profile_with_shells_df.to_parquet(sc_profile_output_path, index=False)
sc_profile_with_shells_df.head()

,object_id,image_set,ParentOrganoid,Nuclei_NoChannel_AreaSizeShape_Volume,Nuclei_NoChannel_AreaSizeShape_CenterX,Nuclei_NoChannel_AreaSizeShape_CenterY,Nuclei_NoChannel_AreaSizeShape_CenterZ,Nuclei_NoChannel_AreaSizeShape_BboxVolume,Nuclei_NoChannel_AreaSizeShape_MinX,Nuclei_NoChannel_AreaSizeShape_MaxX,...,Cytoplasm_ER_Texture_InverseDifferenceMoment-256-3,Cytoplasm_ER_Texture_SumAverage-256-3,Cytoplasm_ER_Texture_SumEntropy-256-3,Cytoplasm_ER_Texture_SumVariance-256-3,Cytoplasm_ER_Texture_Variance-256-3,Nuclei_NoChannel_Neighbors_ShellAssignments,Nuclei_NoChannel_Neighbors_DistancesFromCenter,Nuclei_NoChannel_Neighbors_DistancesFromExterior,Nuclei_NoChannel_Neighbors_NormalizedDistancesFromCenter,Nuclei_NoChannel_Neighbors_ShellsUsed
0,1,C4-2,1,85770.0,504.413734,254.199312,4.221161,128700.0,453,563,...,0.996003,0.815758,0.078643,111.317248,30.190876,3,2.055681,0.517524,0.798880,4
1,2,C4-2,1,60278.0,400.763960,695.127625,4.473689,88320.0,355,447,...,0.995467,0.909430,0.089578,120.021413,31.720237,2,1.809303,0.763901,0.703132,4
2,3,C4-2,1,53927.0,573.770171,885.107386,3.446789,71442.0,501,648,...,0.991875,2.984890,0.168557,617.977303,158.955682,3,2.056000,0.517205,0.799004,4
3,4,C4-2,1,83239.0,742.896923,386.984202,4.943344,116748.0,671,812,...,0.998705,0.612417,0.032869,151.944828,41.983714,1,1.273888,1.299317,0.495059,4
4,5,C4-2,1,138961.0,469.865394,554.114248,7.751455,440640.0,349,553,...,0.995557,0.917593,0.093115,116.811627,32.093011,1,1.171056,1.402149,0.455096,4


In [25]:
nucleocentric_df.to_parquet(nucleocentric_profile_output_path, index=False)
nucleocentric_df.head()

,object_id,image_set,Nucleocentric_ER_CHAMMI75_Feature0,Nucleocentric_ER_CHAMMI75_Feature1,Nucleocentric_ER_CHAMMI75_Feature10,Nucleocentric_ER_CHAMMI75_Feature100,Nucleocentric_ER_CHAMMI75_Feature101,Nucleocentric_ER_CHAMMI75_Feature102,Nucleocentric_ER_CHAMMI75_Feature103,Nucleocentric_ER_CHAMMI75_Feature104,...,Nucleocentric_DNA_SAMMed3D_Feature91,Nucleocentric_DNA_SAMMed3D_Feature92,Nucleocentric_DNA_SAMMed3D_Feature93,Nucleocentric_DNA_SAMMed3D_Feature94,Nucleocentric_DNA_SAMMed3D_Feature95,Nucleocentric_DNA_SAMMed3D_Feature96,Nucleocentric_DNA_SAMMed3D_Feature97,Nucleocentric_DNA_SAMMed3D_Feature98,Nucleocentric_DNA_SAMMed3D_Feature99,ParentOrganoid
0,1,C4-2,-1.816356,-3.321203,2.694317,-1.820192,-2.201500,-1.601819,5.214017,-3.045990,...,-0.102242,0.048619,-0.010611,0.012938,-0.069889,-0.100610,0.247914,0.397652,0.202819,1
1,2,C4-2,-2.428116,-5.585107,4.525021,0.790354,-1.444500,-1.029147,2.224172,0.819189,...,-0.077406,0.038425,-0.010460,0.004507,-0.015260,-0.060301,0.229517,0.360352,0.173643,1
2,3,C4-2,-1.593336,-1.598165,-1.027514,4.432458,2.816724,-1.617609,0.930458,-1.505103,...,0.020256,0.058455,-0.010440,0.042484,-0.056840,0.040460,0.292883,0.312393,0.295259,1
3,4,C4-2,-2.242343,-0.089431,-1.363333,3.002181,-4.259313,2.983073,2.151218,-0.937768,...,-0.072737,0.068186,-0.010723,0.010222,-0.072979,-0.089823,0.264413,0.386310,0.190696,1
4,5,C4-2,0.238446,2.341780,-2.680913,0.148467,1.685059,1.447748,9.391006,-3.827739,...,-0.120807,-0.018729,-0.010395,0.017792,-0.049804,-0.072775,0.147058,0.317047,0.167767,1
